<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="images/thumbnails/mcp-arxiv-hero.svg" align="center" width="35%">
</div>

<br>

# BUILDING MCP SERVERS

<br>

**About:** This notebook teaches the server side of tool-calling with large language models. You will define tool functions, describe them with JSON schemas, wire them into a dispatcher, and execute them safely inside a request loop.

**Learning Goals:** Define tool functions the model can call, write JSON schemas that describe their inputs, build a dispatcher that maps schema requests to Python callables, and execute those tools safely with model-provided arguments.

**Keywords:** mcp, tool-calling, function-calling, json-schema, dispatcher, anthropic-api

**Prerequisite Knowledge:** (1) Python (functions, dicts, type hints), (2) JSON structure and syntax, (3) reading API documentation

**Target User:** Developers who want to give an LLM the ability to take actions - search a database, call a web API, run a computation - rather than only produce text.


<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>


#### CONTENTS

> #### [PART 0: WHY TOOL-CALLING MATTERS](#Part_0)
> #### [PART 1: DEFINING TOOL FUNCTIONS](#Part_1)
> #### [PART 2: WRITING TOOL SCHEMAS](#Part_2)
> #### [PART 3: BUILDING A TOOL DISPATCHER](#Part_3)
> #### [PART 4: EXECUTING AND TESTING TOOLS](#Part_4)

#### APPENDIX

> #### [COMPANION MATERIALS](#Appendix_1)
> #### [REFERENCES AND FURTHER READING](#Appendix_2)

<br>


<a id='Part_0'></a>

<hr style="border: 2px solid#003262;" />

#### PART 0

## **WHY** TOOL-CALLING **MATTERS**


<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/tool-anatomy.svg" align="center" width="60%" padding="10"><br>
    <br>
    Anatomy of an MCP tool: schema, function, and dispatcher working together.
</div>

<br>

Language models reason and write fluent text, but they cannot query a database, hit an HTTP endpoint, or read a file on their own. **Tool-calling** (also called function-calling) closes that gap: you register a set of callable operations, the model decides *when* to invoke them, and your code actually runs them.

The interaction is a strict contract. The model emits a structured request - a tool name plus arguments that match a schema you published. Your program dispatches the call, executes the function, and hands the result back into the conversation. The model reads the result and decides what to do next - answer the user, call another tool, or refine its argument choice.

This notebook builds the **server side** of that contract: the pieces that live in your process and are triggered by the model.

___

**Note:** The Model Context Protocol (MCP) formalizes this pattern across LLM providers. The code in this notebook uses Anthropic's Messages API (`tools=[...]`), but the design - schemas, dispatcher, execution loop - transfers to OpenAI function-calling and Google's function-calling APIs with only surface-level renames. See the [MCP specification](https://modelcontextprotocol.io/) for the cross-provider standard.

___


#### **A concrete example: searching arXiv**

An LLM chatbot needs to search academic papers. Rather than hard-code the search behavior into the model's prompt, we register two tools:

- `search_papers(topic, max_results)` - queries arXiv and stores metadata about matching papers
- `extract_info(paper_id)` - retrieves stored metadata for one paper by its arXiv ID

The model decides *when* to call each one and *what* arguments to pass. Your code decides what happens when it runs.

We use this pair as the running example through the entire notebook - by Part 4 you will have implemented both, wired them into a dispatcher, and tested them in isolation.


<!--Navigate back to table of contents-->
<div align="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->


<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **DEFINING** TOOL **FUNCTIONS**


Tool functions are ordinary Python functions. The features that make them work as *tools* have nothing to do with any framework - they are just three properties you should maintain:

1. **A clear signature** - type-annotated arguments, a documented return type. The schema in Part 2 mirrors this signature; if the two drift apart the model will send arguments your function cannot accept.
2. **Deterministic behavior, or documented side effects** - the model reasons about the *result* of calling your tool. Tools that silently mutate state confuse that reasoning.
3. **Explicit error signals** - return an error *value*, do not raise. The dispatcher in Part 3 relies on this: a raised exception crashes the whole conversation, a returned error string lets the model apologize and try again.

Below we implement the two arXiv tools with those properties.


#### **1.1 `search_papers` - query arXiv and persist metadata**
___


In [ ]:
import arxiv
import json
import os
from typing import List

PAPER_DIR = "papers"


def search_papers(topic: str, max_results: int = 5) -> List[str]:
    """
    Search arXiv for papers on a topic and persist their metadata.

    Args:
        topic: Free-text topic to search for.
        max_results: Maximum number of papers to retrieve. Defaults to 5.

    Returns:
        List of arXiv short IDs (e.g., "2501.12345v1") for the papers found.

    Side effects:
        Creates PAPER_DIR/<topic_slug>/papers_info.json and merges new results
        into any existing entries there.
    """
    client_arxiv = arxiv.Client()

    search = arxiv.Search(
        query=topic,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance,
    )
    papers = client_arxiv.results(search)

    path = os.path.join(PAPER_DIR, topic.lower().replace(" ", "_"))
    os.makedirs(path, exist_ok=True)
    file_path = os.path.join(path, "papers_info.json")

    try:
        with open(file_path, "r") as f:
            papers_info = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        papers_info = {}

    paper_ids = []
    for paper in papers:
        paper_id = paper.get_short_id()
        paper_ids.append(paper_id)
        papers_info[paper_id] = {
            "title": paper.title,
            "authors": [author.name for author in paper.authors],
            "summary": paper.summary,
            "pdf_url": paper.pdf_url,
            "published": str(paper.published.date()),
        }

    with open(file_path, "w") as f:
        json.dump(papers_info, f, indent=2)

    return paper_ids


The function has a **documented side effect** (writing to disk) rather than a hidden one. Callers - the dispatcher and, indirectly, the model - can reason about what happens beyond the return value.

The return type is `List[str]` rather than a rich object. That is deliberate: the model reads text, not Python. A list of IDs serializes cleanly into a comma-separated string the model can then feed into `extract_info`. Full metadata lives on disk so it is available when the model asks for it, but not so bulky that it fills the context window on every search.

___

**Note:** `arxiv.SortCriterion.Relevance` and `arxiv.Client()` are verified against the `arxiv` Python client 2.x API. Re-check at [lukasschwab/arxiv.py](https://github.com/lukasschwab/arxiv.py) if you upgrade the package.

___


#### **1.2 `extract_info` - retrieve stored metadata by ID**
___


In [ ]:
def extract_info(paper_id: str) -> str:
    """
    Look up stored metadata for a specific paper.

    Args:
        paper_id: arXiv short ID (e.g., "2501.12345v1").

    Returns:
        JSON-formatted string with the paper's metadata if found; otherwise a
        human-readable error message.
    """
    for topic_dir in os.listdir(PAPER_DIR):
        topic_path = os.path.join(PAPER_DIR, topic_dir)
        if not os.path.isdir(topic_path):
            continue

        info_file = os.path.join(topic_path, "papers_info.json")
        if not os.path.isfile(info_file):
            continue

        try:
            with open(info_file, "r") as f:
                papers_info = json.load(f)
        except (FileNotFoundError, json.JSONDecodeError):
            continue

        if paper_id in papers_info:
            return json.dumps(papers_info[paper_id], indent=2)

    return f"There's no saved information related to paper {paper_id}."


Two design choices in `extract_info` matter:

- **Returns a string, never raises.** If the paper is not on disk, the function returns a sentence the model can read and react to. If we raised `KeyError` instead, the model would see nothing except a crashed dispatcher.
- **Returns JSON, not a `dict`.** The final consumer is the model, which reads text. Returning JSON is one less type conversion for the dispatcher to worry about.


<!--Concept Check banner-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left">
            <strong>CONCEPT</strong> CHECK
        </a>
    </span>
</div>
<!-------------------------------------->


> **Question:** `extract_info` returns a JSON string rather than a Python `dict`. In one or two sentences, explain the design reason for that choice.

<br>

```python
# Write your answer as a comment.
# Hint: think about who reads the return value and what that reader can process.
```

<hr style="border: 2px solid#003262;" />


<!--Navigate back to table of contents-->
<div align="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->


<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **WRITING** TOOL **SCHEMAS**


The model cannot call your Python function directly - it has never seen your codebase. Instead, you publish a **schema** describing each tool: its name, its purpose, and what inputs it accepts. The model uses the schema in two ways.

- **Selection.** The `description` field is how the model decides *whether* to call the tool at all. A vague description ("search things") gets called for queries where you did not want it; a specific description ("search arXiv for academic papers on a topic") gets called when it should.
- **Argument construction.** The `input_schema` (a JSON Schema fragment) tells the model which arguments to pass and what types they must be. Required arguments listed here will not be omitted.

The schema is a **contract** between your code and the model. Everything the model does with your tools starts by reading this.


#### **2.1 Schema for `search_papers`**
___


In [ ]:
search_papers_schema = {
    "name": "search_papers",
    "description": (
        "Search arXiv for academic papers on a topic and persist their metadata. "
        "Returns a list of arXiv paper IDs that can be passed to extract_info."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "topic": {
                "type": "string",
                "description": "Free-text topic to search for (e.g., 'quantum error correction', 'retrieval augmented generation')."
            },
            "max_results": {
                "type": "integer",
                "description": "Maximum number of results to retrieve.",
                "default": 5
            }
        },
        "required": ["topic"]
    }
}


#### **2.2 Schema for `extract_info`**
___


In [ ]:
extract_info_schema = {
    "name": "extract_info",
    "description": (
        "Retrieve stored metadata for one paper by its arXiv short ID. "
        "Call this only after search_papers has produced the ID."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "paper_id": {
                "type": "string",
                "description": "arXiv short ID as returned by search_papers, e.g., '2501.12345v1'."
            }
        },
        "required": ["paper_id"]
    }
}

tools = [search_papers_schema, extract_info_schema]

for t in tools:
    print(f"{t['name']}: {t['description'][:70]}...")


<br>

**Schema quality checklist**

- **Specific descriptions.** "Search arXiv for academic papers" tells the model when to reach for the tool. "A general search" does not.
- **`required` only lists what has no default.** `max_results` has a default, so it is optional; `topic` does not, so it is required.
- **Types match the Python signature.** If your function takes `int` and your schema says `string`, the model will send `"5"` and the function will crash on the arithmetic.
- **Return values are described in prose, not schema.** JSON Schema in the Anthropic API describes *input*; return semantics go in the tool `description`.

___

**Note:** Anthropic's Messages API tool schema format (top-level `name`, `description`, `input_schema`) is documented at [Anthropic Tool Use docs](https://docs.anthropic.com/en/docs/build-with-claude/tool-use). Verified against the API as of 2026-08. OpenAI and Google use analogous but not identical shapes.

___


<!--Concept Check banner-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left">
            <strong>CONCEPT</strong> CHECK
        </a>
    </span>
</div>
<!-------------------------------------->


> **Question:** A junior colleague writes `"description": "A search tool"` for `search_papers`. Explain in one sentence what the model is likely to do wrong at inference time, and rewrite the description so the same mistake is unlikely.

<br>

```python
# Draft the improved description as a Python string.
better_description = "..."
```

<hr style="border: 2px solid#003262;" />


<!--Navigate back to table of contents-->
<div align="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->


<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **BUILDING** A TOOL **DISPATCHER**


When the model requests a tool, the API sends you back the tool *name* as a string and the arguments as a `dict`. Somewhere in your code you must map "the model asked for `search_papers`" to "call the Python callable named `search_papers`". That mapping is the **dispatcher**.

The dispatcher does three jobs:

1. Look up the callable by name.
2. Invoke it with the provided arguments.
3. Normalize whatever it returns into a string the model can read.

Every one of these has a failure mode - unknown tool, bad argument shape, non-serializable return value. Part 4 covers how to handle them; Part 3 builds the happy path first.


#### **3.1 A registry of callables**
___


In [ ]:
tool_functions = {
    "search_papers": search_papers,
    "extract_info": extract_info,
}

for name, fn in tool_functions.items():
    print(f"  {name} -> {fn.__module__}.{fn.__name__}")


The registry is a plain `dict` mapping tool names (as they appear in the schema) to callables (as they exist in your process). Adding a new tool is a matter of writing the function, writing its schema, and adding one line here. Part 4 of Notebook 03 demonstrates that extension.


#### **3.2 The `execute_tool` function**
___


In [ ]:
def execute_tool(tool_name: str, tool_args: dict) -> str:
    """
    Run a registered tool and return its result as a string.

    Args:
        tool_name: Name of a tool present in tool_functions.
        tool_args: Keyword arguments to pass to the tool.

    Returns:
        A string suitable for feeding back into the model as a tool_result.
        None becomes a completion message; lists become comma-separated strings;
        dicts become pretty-printed JSON; everything else is str()'d.

    Raises:
        KeyError: If tool_name is not registered. Part 4 wraps this raise so the
                  model sees a readable error instead of a crashed process.
    """
    if tool_name not in tool_functions:
        raise KeyError(f"Unknown tool: {tool_name!r}")

    result = tool_functions[tool_name](**tool_args)

    if result is None:
        return "The operation completed but did not return any results."
    if isinstance(result, list):
        return ", ".join(str(x) for x in result)
    if isinstance(result, dict):
        return json.dumps(result, indent=2)
    return str(result)


The normalization matters. The model reads *text* - it cannot consume a Python `list` or `dict` object. If your dispatcher returns a bare object the API will refuse the message. Explicit normalization keeps that concern in one place instead of scattering `json.dumps` calls through every tool.


#### **3.3 A dry-run of the dispatch**
___


In [ ]:
# We simulate what the model would send back so this cell runs without hitting the API.
simulated_tool_name = "extract_info"
simulated_tool_args = {"paper_id": "nonexistent_id"}

result = execute_tool(simulated_tool_name, simulated_tool_args)
print(f"Dispatcher returned:\n{result}")


The unknown-paper case returns the sentence we designed in Part 1. That sentence is what the model will read. If we had raised an exception instead, the conversation would have crashed before the model got a chance to say "I could not find that paper - want me to search first?"


<!--Concept Check banner-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left">
            <strong>CONCEPT</strong> CHECK
        </a>
    </span>
</div>
<!-------------------------------------->


> **Question:** Extend `execute_tool` so that if `tool_args` contains an unexpected keyword the function does not raise `TypeError`. Instead it should return a string starting with `"Error calling <tool_name>:"` that mentions the bad argument. Write the extended version below.

<br>

```python
def execute_tool_safe(tool_name: str, tool_args: dict) -> str:
    ### YOUR CODE HERE ###
    ...
```

<hr style="border: 2px solid#003262;" />


<!--Navigate back to table of contents-->
<div align="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->


<a id='Part_4'></a>

<hr style="border: 2px solid#003262;" />

#### PART 4

## **EXECUTING** AND **TESTING** TOOLS


The dispatcher runs one tool. The **conversation loop** decides when to call the dispatcher, what to do with the result, and when to stop. This is where server-side and client-side concerns meet: your server code is invoked from a client-side loop that sends messages, receives responses, and threads tool results back into the conversation.

The full client loop lives in Notebook 02. Here we look at the *shape* of the loop from the server's perspective - what a well-written tool needs to fit into it - and then how to test tools before they ever run inside that loop.


#### **4.1 The execution loop, sketched**
___


```
1. User sends a query.
2. Client posts (query, tool schemas) to the model.
3. Model responds with content blocks. For each block:
     - text -> show to user
     - tool_use -> dispatcher runs the tool, result posted back to model
4. Repeat until the model returns a response with no tool_use blocks.
```

For your tool to fit cleanly in step 3, it must:

- **Return promptly.** A tool that hangs stalls the entire conversation. Set timeouts on outbound HTTP or database calls.
- **Never crash.** Wrap third-party calls that can raise, and translate exceptions into return values.
- **Bound its output size.** A tool that returns a 100 KB blob will blow the model's context window on the next request.


#### **4.2 A minimal end-to-end call**
___


In [ ]:
from dotenv import load_dotenv
import anthropic

load_dotenv()
api_key = os.getenv("ANTHROPIC_API_KEY")

# The rest of this cell is inert unless ANTHROPIC_API_KEY is set.
# We skip the actual API call here to keep the notebook offline-runnable.
if api_key:
    client = anthropic.Anthropic(api_key=api_key)
    print("Anthropic client initialized. See arxiv_chatbot.py for the full loop.")
else:
    print("ANTHROPIC_API_KEY not set. Skipping live call - see arxiv_chatbot.py.")


The full loop implementation is in `arxiv_chatbot.py`. Notebook 02 (`02_building_mcp_clients.ipynb`) walks through it cell-by-cell. Notebook 03 (`03_mcp_in_practice.ipynb`) traces a specific query end-to-end through the same code.

___

**Note:** Model identifier strings (`claude-sonnet-4-6`, `claude-opus-4-7`, etc.) are volatile - Anthropic retires older Claude versions periodically. Before pinning a model in production, check the current list at [docs.anthropic.com/en/docs/about-claude/models](https://docs.anthropic.com/en/docs/about-claude/models). The chatbot script currently defaults to `claude-3-7-sonnet-20250219`; update it when you upgrade.

___


#### **4.3 Testing tools in isolation**
___

The fastest way to ship broken tools is to only ever test them through the model. Model responses are non-deterministic, so a failure at the API level tells you nothing about which layer is wrong. Unit-test each tool at the Python level first; the model integration test only proves that the schema description is good enough for the model to *find* the tool.


In [ ]:
def test_dispatcher_unknown_tool():
    """Unknown tool names surface as KeyError so the safe wrapper can catch them."""
    raised = False
    try:
        execute_tool("nonexistent_tool", {})
    except KeyError:
        raised = True
    assert raised, "Dispatcher should raise KeyError on unknown tools"
    print("PASS: dispatcher raises KeyError on unknown tool")


def test_dispatcher_normalizes_list():
    """List returns are joined into a comma-separated string."""
    # Register a temporary callable so this test doesn't hit arXiv.
    tool_functions["_test_list"] = lambda: ["a", "b", "c"]
    try:
        result = execute_tool("_test_list", {})
        assert result == "a, b, c", f"Expected 'a, b, c', got {result!r}"
        print("PASS: list return is joined")
    finally:
        del tool_functions["_test_list"]


test_dispatcher_unknown_tool()
test_dispatcher_normalizes_list()


These tests exercise the dispatcher without any network call. Notebook 03 shows how to layer end-to-end tests on top; the practical rule is: unit tests catch broken code, integration tests catch broken *schemas*.


<!--Concept Check banner-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left">
            <strong>CONCEPT</strong> CHECK
        </a>
    </span>
</div>
<!-------------------------------------->


> **Question:** Add one more unit test to the suite above that verifies `execute_tool` returns pretty-printed JSON (not a raw `dict.__str__`) when a tool returns a dictionary. Use the same "temporary callable" pattern.

<br>

```python
def test_dispatcher_normalizes_dict():
    ### YOUR CODE HERE ###
    ...

test_dispatcher_normalizes_dict()
```

<hr style="border: 2px solid#003262;" />


<!--Navigate back to table of contents-->
<div align="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->


<a id='Appendix_1'></a>

<hr style="border: 2px solid#003262;" />

#### APPENDIX

## **COMPANION** MATERIALS


- **Homework**: [Building MCP Servers Homework](homework/building_mcp_servers_homework.ipynb) - practice writing tools, schemas, and a dispatcher on an offline dataset (no API key required).
- **Working script**: `arxiv_chatbot.py` - the full server + client implementation this notebook walks through.
- **Next notebook**: [Building MCP Clients](02_building_mcp_clients.ipynb) - the loop that calls your dispatcher.
- **Advanced examples**: [`examples/`](examples/) - async execution, streaming responses, FastAPI wrapper, multi-tool reflection.


<a id='Appendix_2'></a>

<hr style="border: 2px solid#003262;" />

##### **REFERENCES**


- [Model Context Protocol specification](https://modelcontextprotocol.io/)
- [Anthropic Messages API - Tool use](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)
- [Anthropic tool_use content blocks](https://docs.anthropic.com/en/docs/build-with-claude/tool-use#tool-use-and-tool-result-content-blocks)
- [JSON Schema specification](https://json-schema.org/)
- [arxiv Python client](https://github.com/lukasschwab/arxiv.py)


<hr style="border: 6px solid#003262;" />
